In [71]:
class Individual:
    def __init__(self,genome,fitness = 0,validated=False):
        self.genome = genome
        self.fitness = fitness
        self.validated = validated

In [72]:
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import r2_score
from sklearn.metrics import confusion_matrix

In [73]:
path = "../data_base/bbc_text_cls_adapted.csv"

In [74]:
data = pd.read_csv(filepath_or_buffer = path)

In [75]:
labels = data["labels"].unique()

In [76]:
X = data["text"]
y = data["labels"]
labels = y.unique()

In [77]:
if y.dtype == object or y.dtype == str:
    flag = True
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)
    y = pd.DataFrame(y)

In [78]:
columns = data.iloc[:,2:]
columns = columns.apply(le.transform)
columns = columns.apply(lambda linha: list(set(linha)), axis=1)

In [79]:
list_cc = []
list_dc = []

for index,_ in enumerate(columns):
    if len(_) == 1:
        list_cc.append(data.iloc[index])
        # list_index_cc.append(index)
    else:
        list_dc.append(data.iloc[index])
        # list_index_cd(_)

df_cc = pd.DataFrame(list_cc)
df_dc = pd.DataFrame(list_dc)

In [80]:
if df_cc.empty == False:
    y_cc = df_cc['labels']
    X_cc = df_cc['text']
    y_dc = df_dc['labels']
    X_dc = df_dc['text']
else:
    y_dc = df_dc['labels']
    X_dc = df_dc['text']

In [81]:
if df_cc.empty == True:
    X_train, X_test, y_train, y_test = train_test_split(X_dc, y_dc, test_size = 0.33, random_state = 1, stratify = y)
else:
    X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split(X_cc, y_cc, test_size = 0.35, random_state = 1, stratify = y)
    X_train_dc, X_test_dc, y_train_dc, y_test_dc = train_test_split(X_dc, y_dc, test_size = 0.65, random_state = 1, stratify = y)
    
    X_train = pd.concat([X_train_cc,X_train_dc], ignore_index=True)
    X_test = pd.concat([X_test_cc,X_test_dc], ignore_index=True)
    y_train = pd.concat([y_train_cc,y_train_dc], ignore_index=True)
    y_test = pd.concat([y_test_cc,y_test_dc], ignore_index=True)

In [82]:
vectorizer = TfidfVectorizer(stop_words='english')

X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

In [83]:
indices_treino = y_train.index
columns = data.iloc[indices_treino, 2:]

In [84]:
columns = columns.apply(le.transform)

In [85]:
matrix = np.array(columns)

In [86]:
matrix = np.array([list(dict.fromkeys(lista)) for lista in matrix], dtype=object)

In [87]:
len_ = 0
for _ in matrix:
    if 1 < len(_):
        len_+=1

In [88]:
labels = le.transform(labels)

In [89]:
naive_bayes_mult = MultinomialNB()

In [90]:
def fitness_function(genome,matrix,X_train_transformed,a,b,g):
    matrix_ = matrix.copy()
    df_ = pd.DataFrame()
    cont = 0
    saved_dc_idx = []
    saved_cc_idx = []
    dc_sum = 0
    cc_sum = 0

    for index,values in enumerate(matrix_):
        if len(values) > 1:
            matrix_[index] = genome[cont]
            cont+=1
            saved_dc_idx.append((index,matrix_[index]))
        else:
            saved_cc_idx.append((index,matrix_[index]))

    matrix_ = [item[0] if isinstance(item, list) else item for item in matrix_]

    naive_bayes_mult.fit(X_train_transformed,matrix_)
    naive_bayes_mult.predict(X_train_transformed)
    df_ = naive_bayes_mult.predict_log_proba(X_train_transformed)

    for index,values in saved_dc_idx:
        dc_sum += df_[[index],[values]]

    for index,values in saved_cc_idx:
        cc_sum += df_[[index],[values]]

    cont = np.bincount(matrix_)
    variance = np.var(cont)

    #list_before_sum = [max(i) for i in df_]
    #fitness = sum(list_before_sum)

    fitness = a*cc_sum+b*dc_sum-g*variance
    #print(fitness)
    
    return(fitness)

# def fitness_function(individual, matrix, X_train_transformed, a, b, g):
#     matrix_ = matrix.copy()
#     cont = 0
#     saved_dc_idx = []
#     saved_cc_idx = []
    
#     # Garantimos desde o início que essas variáveis são números puros (floats)
#     dc_sum = 0.0
#     cc_sum = 0.0
    
#     genome = np.array(individual.genome)

#     for index, values in enumerate(matrix_):
#         if len(values) > 1:
#             # Forçamos o cast para int para evitar arrays soltos
#             matrix_[index] = int(genome[cont])
#             cont += 1
#             saved_dc_idx.append((int(index), int(matrix_[index])))
#         else:
#             val = values[0] if isinstance(values, list) else values
#             saved_cc_idx.append((int(index), int(val)))

#     # Achata a matriz para o fit
#     matrix_ = [int(item[0]) if isinstance(item, list) else int(item) for item in matrix_]

#     naive_bayes_mult.fit(X_train_transformed, matrix_)
    
#     # Retorna o array 2D
#     df_ = naive_bayes_mult.predict_log_proba(X_train_transformed)

#     # Usa chaves SIMPLES e força a extração do float da matriz
#     for index, val in saved_dc_idx:
#         dc_sum += float(df_[[index], [val]])

#     for index, val in saved_cc_idx:
#         cc_sum += float(df_[[index], [val]])

#     cont_bincount = np.bincount(matrix_)
#     variance = float(np.var(cont_bincount))

#     # O cálculo final será garantidamente um único número float
#     fitness = float(a * cc_sum + b * dc_sum - g * variance)
    
    #return fitness

In [91]:
def create_initial_population(size,len_,labels,matrix,X_train_transformed,a,b,g):
    population = []
    for _ in range(size):
        genome = random.choices(labels, k=len_)
        genome = np.array(genome)
        fitness = fitness_function(genome,matrix,X_train_transformed,a,b,g)
        individual = Individual(genome=genome,fitness=fitness)
        #individual.fitness = fitness
        population.append(individual)
        
    return population

In [92]:
def selection(selected_tournment):
    selected = []
    for _ in range(len(selected_tournment)):
        first_pair = [selected_tournment[0],selected_tournment[1]]
        second_pair = [selected_tournment[2],selected_tournment[3]]
        winner_1 = max(first_pair, key=lambda x: x.fitness)
        winner_2 = max(second_pair, key=lambda x: x.fitness)
        selected.append(winner_1)
        selected.append(winner_2)
    return selected

In [93]:
def genome_decision(genome_1,genome_2):
    genome_1 = list(genome_1)
    genome_2 = list(genome_2)

    len_ = len(genome_1)
    
    decision_parent = []
    genome_child = []
    
    for ind in range(len_):
        decision_parent.append(random.randint(0,1))

    for ind in range(len_):
        if decision_parent[ind] == 1:
            genome_child.append(genome_1[ind])
        else:
            genome_child.append(genome_2[ind])

    genome_child = np.array(genome_child)

    return(genome_child)

In [94]:
def crossover(selected_1, selected_2,matrix,X_train_transformed,a,b,g):
    
    child_1 = Individual(genome = genome_decision(selected_1.genome,selected_2.genome))
    child_2 = Individual(genome = genome_decision(selected_2.genome,selected_1.genome))
    
    fitness_1 = fitness_function(child_1.genome, matrix, X_train_transformed,a,b,g)
    fitness_2 = fitness_function(child_2.genome, matrix, X_train_transformed,a,b,g)

    child_1.fitness = fitness_1
    child_2.fitness = fitness_2
    
    return child_1, child_2

In [108]:
def mutation(child, mutation_rate,labels,matrix,X_train_transformed,a,b,g):
    genome = child.genome
    child.validated = False
    
    for i in range(len(genome)):
        if random.random() < mutation_rate:
            sorted_index = random.randrange(len(genome))
            if(genome[sorted_index]==1):
                genome[sorted_index] = random.choice(labels)
                
    genome = np.array(genome)
    print(genome)
    child.genome = genome
    child.fitness = fitness_function(child.genome,matrix,X_train_transformed,a,b,g)
    
    return(child)

In [96]:
def select_individuals(population):
    available = range(len(population))
    sorted_parents = random.sample(available,4)

    parent1 = population[sorted_parents[0]]
    parent2 = population[sorted_parents[1]]
    parent3 = population[sorted_parents[2]]
    parent4 = population[sorted_parents[3]]

    selected_tournment = [parent1,parent2,parent3,parent4]
    
    selected_crossover = selection(selected_tournment)

    return(selected_crossover)

In [97]:
# Main genetic algorithm function
def genetic_algorithm(population_size,generations, mutation_rate,labels,len_,matrix,X_train_transformed,a,b,g):
    population = create_initial_population(population_size,len_,labels,matrix,X_train_transformed,a,b,g)

    for generation in range(generations):
        new_population = []

        best_individual = max(population, key=lambda x: fitness_function(x.genome, matrix, X_train_transformed,a,b,g))
        best_fitness = fitness_function(best_individual.genome, matrix, X_train_transformed,a,b,g)

        index_best_individuals = sorted(range(len(population)), key=lambda x: population[x].fitness)
        primary_best_individual = population[index_best_individuals[0]]
        second_best_individual = population[index_best_individuals[1]]

        for _ in range(int((population_size-2)/2)):

            selected_crossover = select_individuals(population)
    
            child_1, child_2 = crossover(selected_crossover[0], selected_crossover[1], matrix, X_train_transformed,a,b,g)
    
            child_1 = mutation(child_1, mutation_rate, labels, matrix, X_train_transformed,a,b,g)
            child_2 = mutation(child_2, mutation_rate, labels, matrix, X_train_transformed,a,b,g)
            
            # sorted_index = random.sample(range(len(population)),2)
            # print(sorted_index)
            # population[sorted_index[0]] = child1
            # population[sorted_index[1]] = child2
            
            # index_worst_individuals = sorted(range(len(population)), key=lambda x: population[x].value)
            new_population.append(child_1)
            new_population.append(child_2)

        new_population.append(primary_best_individual)
        new_population.append(second_best_individual)

        population = new_population

    return (best_individual)

In [98]:
a = 1
b = 1
g = 1

In [109]:
best_individual = genetic_algorithm(20,50,1,labels,len_,matrix,X_train_transformed,a,b,g)

[3 0 3 ... 2 3 3]
[3 0 4 ... 0 3 4]
[4 0 2 ... 4 2 4]
[4 4 2 ... 4 0 1]
[4 0 1 ... 4 2 2]
[3 4 4 ... 0 2 0]
[1 1 0 ... 2 2 3]
[3 3 3 ... 2 2 2]
[0 2 3 ... 2 3 0]
[2 0 3 ... 4 2 3]
[3 4 1 ... 3 2 2]
[3 4 2 ... 1 2 3]
[0 4 1 ... 0 0 2]
[4 4 1 ... 0 0 2]
[4 1 4 ... 2 2 3]
[4 4 4 ... 2 2 4]
[3 2 2 ... 0 3 3]
[0 0 3 ... 2 1 3]
[3 2 2 ... 2 3 3]
[4 1 2 ... 2 3 3]
[4 4 0 ... 2 0 3]
[4 3 2 ... 2 0 3]
[3 4 3 ... 0 2 3]
[0 4 3 ... 2 2 0]
[4 3 3 ... 2 0 3]
[0 0 0 ... 2 0 3]
[4 0 3 ... 2 0 1]
[4 4 3 ... 2 3 1]
[3 0 2 ... 2 3 3]
[4 4 2 ... 4 3 0]
[3 0 3 ... 2 3 0]
[3 0 3 ... 2 3 0]
[4 4 2 ... 4 0 0]
[4 4 2 ... 4 2 0]
[4 4 2 ... 2 0 4]
[4 4 2 ... 2 0 2]
[4 2 2 ... 2 0 3]
[4 3 2 ... 2 3 3]
[3 4 3 ... 2 2 2]
[4 4 3 ... 2 2 3]
[3 4 2 ... 4 2 3]
[3 4 2 ... 2 2 3]
[3 0 2 ... 2 2 3]
[3 4 3 ... 2 3 0]
[4 0 2 ... 4 0 4]
[4 4 2 ... 2 0 4]
[4 3 2 ... 2 0 3]
[4 3 2 ... 2 0 3]
[4 4 2 ... 2 3 3]
[4 4 2 ... 4 3 0]
[4 4 2 ... 2 0 0]
[4 4 2 ... 2 2 4]
[3 3 3 ... 2 0 3]
[4 3 3 ... 2 0 3]
[4 4 3 ... 2 2 3]
[4 0 2 ...

IndexError: index 4 is out of bounds for axis 1 with size 4

In [ ]:
genome = le.inverse_transform(best_individual.genome)

In [ ]:
matrix_ = matrix.copy()

#df_ = pd.DataFrame()
cont = 0
for index,values in enumerate(matrix_):
    if len(values) > 1:
        matrix_[index] = genome[cont]
        cont+=1

matrix_ = [item[0] if isinstance(item, list) else item for item in matrix_]

In [ ]:
if flag == True:
    matrix_ = le.inverse_transform(matrix_)
    y_train = le.inverse_transform(y_train)

In [ ]:
confusion_matrix(y_train, matrix_)